# TechNest Sample Data Generator

This notebook fills the TechNest MySQL database with sample data for testing the project's SQL queries. Faker provides realistic names and addresses, fixed random seeds keep the random values consistent, and planted test cases deliberately break specific business rules so each query has a known expected result.

**Before you run it:**
- Create the tables first by running `sql/01_create_tables.sql`.
- **This notebook deletes every existing row** before inserting new data. If you want to keep your current data, export it first.
- Dates are generated relative to the day you run it, so regenerated data won't match the presentation's screenshots exactly. The infraction and training steps print the IDs of their test-case employees.

## Setup
Import the libraries and connect to the local MySQL server. `getpass()` asks for the password without displaying or saving it.

In [ ]:
import random
from datetime import date, datetime, timedelta
from getpass import getpass

import mysql.connector
from faker import Faker

In [ ]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=getpass("MySQL password: "),
    database="technest"
)

cursor = conn.cursor()
print(conn.is_connected())

## Settings
Fixed seeds keep the random values consistent, and `TODAY` anchors every generated date.

In [ ]:
random.seed(42)
Faker.seed(42)
fake = Faker()

TODAY = date.today()

## Reset the tables
Clear every table so the notebook can be rerun from a clean start. Foreign key checks are paused so the tables can be emptied in any order.

In [ ]:
tables = [
    "customers",
    "discounttiers",
    "employeeinfractions",
    "employees",
    "infractiontypes",
    "lineitems",
    "orders",
    "products",
    "training",
    "trainingcourse"
]

try:
    cursor.execute("SET FOREIGN_KEY_CHECKS = 0")

    for name in tables:
        cursor.execute(f"TRUNCATE TABLE `{name}`")

finally:
    cursor.execute("SET FOREIGN_KEY_CHECKS = 1")

conn.commit()

## Lookup tables
Discount tiers (Rule 25), infraction types and their point values (Rule 8), and training courses (Rule 5).

In [ ]:
sql = """
INSERT INTO discounttiers (MinPoints, DiscountPercent)
VALUES (%s, %s)
"""

rows = [
    (0, 0),
    (100, 5),
    (500, 10),
    (1000, 15),
    (2000, 20),
    (2500, 25),
    (3000, 30)
]

cursor.executemany(sql, rows)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM discounttiers")
print(cursor.fetchone())

In [ ]:
sql = """
INSERT INTO infractiontypes (InfractionType, Points)
VALUES (%s, %s)
"""

rows = [
    ("Late Arrival (15 min)", 1),
    ("Late Arrival (30 min)", 2),
    ("Dress Code Violation", 2),
    ("Unexcused Absence", 5),
    ("No-Call/No-Show", 6)
]

cursor.executemany(sql, rows)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM infractiontypes")
print(cursor.fetchone())

In [ ]:
sql = """
INSERT INTO trainingcourse (CourseType, CourseDescription)
VALUES (%s, %s)
"""

rows = [
    (
        "Basic Enrollment Training",
        "Introduces new employees to company policies and basic job responsibilities."
    ),
    (
        "Customer Service Excellence",
        "Teaches employees how to assist customers and resolve common concerns."
    ),
    (
        "Product Knowledge",
        "Explains product features so employees can answer customer questions."
    )
]

cursor.executemany(sql, rows)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM trainingcourse")
print(cursor.fetchone())

## Customers
27 random customers plus planted test cases: Customer 1 is the only nonmember (Rule 24), and Customers 28 to 30 have 2,200, 2,800, and 3,000 loyalty points (Rule 18).

In [ ]:
sql = """
INSERT INTO customers
(FName, LName, Email, Phone, Address, DOB,
 LoyaltyPoints, Subscription, Registration)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

rows = []

phone_numbers = random.sample(range(2000, 10000), 30)

for i in range(27):
    first = fake.first_name()
    last = fake.last_name()

    email = f"{first.lower()}.{last.lower()}{i}@example.com"
    phone = f"(412) 555-{phone_numbers[i]}"
    address = fake.street_address()

    dob = TODAY - timedelta(days=random.randint(18 * 365, 75 * 365))
    points = random.randint(0, 1999)
    subscription = 1
    registration = TODAY - timedelta(days=random.randint(400, 2000))

    if i == 0:
        subscription = 0

    rows.append((
        first, last, email, phone, address, dob,
        points, subscription, registration
    ))

# Test customer: above the notification threshold
rows.append((
    "Alex", "Morgan", "alex.morgan.test@example.com",
    f"(412) 555-{phone_numbers[27]}",
    "100 Example Street", date(1995, 4, 12),
    2200, 1, TODAY - timedelta(days=800)
))

# Test customer: approaching the points cap
rows.append((
    "Jordan", "Lee", "jordan.lee.test@example.com",
    f"(412) 555-{phone_numbers[28]}",
    "200 Example Street", date(1990, 8, 20),
    2800, 1, TODAY - timedelta(days=1000)
))

# Test customer: exactly at the points cap
rows.append((
    "Taylor", "Reed", "taylor.reed.test@example.com",
    f"(412) 555-{phone_numbers[29]}",
    "300 Example Street", date(1985, 11, 5),
    3000, 1, TODAY - timedelta(days=1200)
))

cursor.executemany(sql, rows)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM customers")
print(cursor.fetchone())

## Employees
20 random employees plus planted cases: two trainers (Rule 5), a new hire with sick time (Rule 10), and two minors, one full-time and one part-time (Rule 15).

In [ ]:
sql = """
INSERT INTO employees
(FName, LName, Email, Phone, Address, DOB, DateHired,
 Department, Position, EmploymentType,
 SickTime, VacationTime, PersonalTime)
VALUES (%s, %s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s, %s)
"""

departments = {
    "Sales": [
        "Sales Associate",
        "Senior Sales Associate"
    ],
    "Customer Service": [
        "Customer Service Representative",
        "Senior Customer Service Representative"
    ],
    "Warehouse": ["Stock Associate"],
    "Management": ["Store Manager"]
}

rows = []

phone_numbers = random.sample(range(100, 200), 25)

# Generate 20 random adult employees.
for i in range(20):
    first = fake.first_name()
    last = fake.last_name()

    email = f"{first.lower()}.{last.lower()}{i}@technest.com"
    phone = f"(412) 555-{phone_numbers[i]:04d}"
    address = fake.street_address()

    age = random.randint(20, 60)
    dob = date(TODAY.year - age, 1, 1)

    # Hire dates must be on or after their 18th birthday.
    eighteenth_birthday = date(dob.year + 18, dob.month, dob.day)

    earliest_hire = max(
        TODAY - timedelta(days=20 * 365),
        eighteenth_birthday
    )

    latest_hire = TODAY - timedelta(days=365)

    days_available = (latest_hire - earliest_hire).days

    date_hired = earliest_hire + timedelta(
        days=random.randint(0, days_available)
    )

    department = random.choice(list(departments.keys()))
    position = random.choice(departments[department])

    employment_type = random.choices(
        ["Full-Time", "Part-Time"],
        weights=[80, 20]
    )[0]

    sick_time = random.randint(0, 9)
    vacation_time = random.randint(0, 14)
    personal_time = random.randint(0, 7)

    rows.append((
        first, last, email, phone, address, dob, date_hired,
        department, position, employment_type,
        sick_time, vacation_time, personal_time
    ))

# Trainer 1
rows.append((
    "Jamie", "Parker", "jamie.parker.trainer@technest.com",
    f"(412) 555-{phone_numbers[20]:04d}",
    "400 Example Street",
    date(TODAY.year - 45, 1, 1),
    TODAY - timedelta(days=12 * 365),
    "Training", "Trainer", "Full-Time",
    5, 10, 3
))

# Trainer 2
rows.append((
    "Casey", "Brooks", "casey.brooks.trainer@technest.com",
    f"(412) 555-{phone_numbers[21]:04d}",
    "500 Example Street",
    date(TODAY.year - 40, 1, 1),
    TODAY - timedelta(days=10 * 365),
    "Training", "Trainer", "Full-Time",
    4, 8, 2
))

# New hire with sick time: intentional Rule 10 violation.
rows.append((
    "Riley", "Bennett", "riley.bennett.newhire@technest.com",
    f"(412) 555-{phone_numbers[22]:04d}",
    "600 Example Street",
    date(TODAY.year - 25, 1, 1),
    TODAY - timedelta(days=60),
    "Sales", "Sales Associate", "Full-Time",
    2, 0, 0
))

# Sixteen-year-old full-time employee: intentional Rule 15 violation.
rows.append((
    "Avery", "Collins", "avery.collins.minor@technest.com",
    f"(412) 555-{phone_numbers[23]:04d}",
    "700 Example Street",
    date(TODAY.year - 16, 1, 1),
    TODAY - timedelta(days=10),
    "Sales", "Sales Associate", "Full-Time",
    0, 0, 0
))

# Fifteen-year-old part-time employee: valid minor test case.
rows.append((
    "Morgan", "Hayes", "morgan.hayes.minor@technest.com",
    f"(412) 555-{phone_numbers[24]:04d}",
    "800 Example Street",
    date(TODAY.year - 15, 1, 1),
    TODAY - timedelta(days=10),
    "Sales", "Sales Associate", "Part-Time",
    0, 0, 0
))

cursor.executemany(sql, rows)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM employees")
print(cursor.fetchone())

## Products
25 products across five categories. The first four start below their reorder thresholds (Rule 22).

In [ ]:
sql = """
INSERT INTO products
(ProductName, Category, Price, Description, Features,
 StockQuantity, ReorderThreshold, LastRestockDate)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

products = [
    ("Smart Office Chair with Speakers", "Chairs", "349.99"),
    ("Heated Ergonomic Chair", "Chairs", "429.99"),
    ("Posture-Tracking Office Chair", "Chairs", "499.99"),
    ("Massage Gaming Chair", "Chairs", "599.99"),
    ("Cooling Mesh Smart Chair", "Chairs", "379.99"),

    ("Self-Cleaning Table", "Tables", "799.99"),
    ("Holo-Projector Coffee Table", "Tables", "1499.99"),
    ("Wireless Charging Side Table", "Tables", "199.99"),
    ("Temperature-Controlled Dining Table", "Tables", "1199.99"),
    ("Touchscreen Coffee Table", "Tables", "999.99"),

    ("Voice-Controlled Standing Desk", "Desks", "699.99"),
    ("Wireless Charging Workstation", "Desks", "549.99"),
    ("Smart Corner Desk", "Desks", "649.99"),
    ("Adjustable Gaming Desk", "Desks", "449.99"),
    ("Compact Motorized Desk", "Desks", "399.99"),

    ("AI-Powered Bed", "Beds", "2499.99"),
    ("Sleep-Tracking Adjustable Bed", "Beds", "1899.99"),
    ("Temperature-Controlled Bed", "Beds", "2199.99"),
    ("Smart Storage Bed", "Beds", "1599.99"),
    ("Voice-Controlled Reclining Bed", "Beds", "1999.99"),

    ("App-Controlled LED Bookshelf", "Storage", "299.99"),
    ("Fingerprint-Lock Filing Cabinet", "Storage", "399.99"),
    ("Smart Charging Nightstand", "Storage", "249.99"),
    ("Motion-Sensor Storage Cabinet", "Storage", "349.99"),
    ("Climate-Controlled Display Cabinet", "Storage", "899.99")
]

category_features = {
    "Chairs": "Adjustable seating supports comfort during work or gaming.",
    "Tables": "Integrated technology adds convenience to everyday surfaces.",
    "Desks": "Built-in technology supports an organized workspace.",
    "Beds": "Smart controls provide a customizable sleeping experience.",
    "Storage": "Integrated technology improves storage convenience."
}

rows = []

for i in range(len(products)):
    name, category, price = products[i]

    description = f"The {name} combines furniture with modern technology."
    features = category_features[category]

    stock_quantity = random.randint(20, 60)
    reorder_threshold = random.randint(10, 20)

    last_restock_date = TODAY - timedelta(
        days=random.randint(0, 60)
    )

    # First four products are below their reorder thresholds.
    if i < 4:
        stock_quantity = random.randint(0, reorder_threshold - 1)

    rows.append((
        name, category, price, description, features,
        stock_quantity, reorder_threshold, last_restock_date
    ))

cursor.executemany(sql, rows)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM products")
print(cursor.fetchone())

## Orders and line items
50 orders spread across the last two years, never on a Saturday. Only members can place orders (Rule 24), and only Sales and Customer Service employees process them. Each order saves the product prices and the customer's loyalty discount at the time of sale (Rule 25), shipped and pickup orders get the matching address and date fields (Rule 19), and the Climate-Controlled Display Cabinet is left out of the last 90 days for the Products Not Selling test.

In [ ]:
# Read existing customers, employees, products, and discount tiers.
cursor.execute("""
SELECT CustomerID, Address, LoyaltyPoints, Registration
FROM customers
WHERE Subscription = 1
ORDER BY CustomerID
""")
customers = cursor.fetchall()

cursor.execute("""
SELECT EmployeeID, DateHired
FROM employees
WHERE Department IN ('Sales', 'Customer Service')
ORDER BY EmployeeID
""")
employees = cursor.fetchall()

cursor.execute("""
SELECT ProductID, Price
FROM products
ORDER BY ProductID
""")
products = cursor.fetchall()

cursor.execute("""
SELECT MinPoints, DiscountPercent
FROM discounttiers
ORDER BY MinPoints
""")
tiers = cursor.fetchall()

# Find the test product by name rather than assuming its ID.
cursor.execute("""
SELECT ProductID
FROM products
WHERE ProductName = 'Climate-Controlled Display Cabinet'
""")
not_selling_product_id = cursor.fetchone()[0]


# Find the discount for the highest qualifying tier.
def get_discount(points):
    discount = 0

    for minimum_points, percent in tiers:
        if points >= minimum_points:
            discount = percent

    return discount


# These ranges do not overlap.
order_periods = [
    (0, 90, 14),
    (91, 180, 12),
    (181, 365, 14),
    (366, 730, 10)
]

order_sql = """
INSERT INTO orders
(CustomerID, EmployeeID, OrderDate, Payment, OrderStatus,
 FulfillmentType, ShippingAddress, DeliveryDate, PickupDate,
 AppliedDiscountPercent)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

lineitem_sql = """
INSERT INTO lineitems
(OrderID, ProductID, Quantity, UnitPrice)
VALUES (%s, %s, %s, %s)
"""

orders_added = 0
lineitems_added = 0
now = datetime.now()

try:
    for minimum_days, maximum_days, number_of_orders in order_periods:
        for i in range(number_of_orders):

            # Choose an order date, excluding Saturdays and future times.
            while True:
                days_ago = random.randint(minimum_days, maximum_days)
                order_day = TODAY - timedelta(days=days_ago)

                order_date = datetime(
                    order_day.year,
                    order_day.month,
                    order_day.day,
                    random.randint(9, 19),
                    random.randint(0, 59)
                )

                if order_day.weekday() != 5 and order_date <= now:
                    break

            # Only use customers registered before the order day.
            eligible_customers = [
                customer for customer in customers
                if customer[3] < order_day
            ]

            # Only use employees hired before the order day.
            eligible_employees = [
                employee for employee in employees
                if employee[1] < order_day
            ]

            if not eligible_customers or not eligible_employees:
                raise ValueError(
                    f"No eligible customer or employee for {order_day}."
                )

            customer = random.choice(eligible_customers)
            employee = random.choice(eligible_employees)

            customer_id = customer[0]
            customer_address = customer[1]
            loyalty_points = customer[2]
            employee_id = employee[0]

            payment = random.choice([
                "Credit Card",
                "Debit Card",
                "Cash"
            ])

            fulfillment_type = random.choice(["Ship", "Pickup"])

            shipping_address = None
            delivery_date = None
            pickup_date = None

            if fulfillment_type == "Ship":
                shipping_address = customer_address

                if days_ago > 7:
                    order_status = "Delivered"
                    delivery_date = order_date + timedelta(
                        days=random.randint(3, 7)
                    )
                else:
                    order_status = "Shipped"

            else:
                if days_ago > 3:
                    order_status = "Picked Up"
                    pickup_date = order_date + timedelta(
                        hours=random.randint(3, 48)
                    )
                else:
                    order_status = "Ready for Pickup"

            discount = get_discount(loyalty_points)

            cursor.execute(order_sql, (
                customer_id,
                employee_id,
                order_date,
                payment,
                order_status,
                fulfillment_type,
                shipping_address,
                delivery_date,
                pickup_date,
                discount
            ))

            order_id = cursor.lastrowid
            orders_added += 1

            # Exclude the test product from recent orders.
            if days_ago <= 90:
                available_products = [
                    product for product in products
                    if product[0] != not_selling_product_id
                ]
            else:
                available_products = products

            selected_products = random.sample(
                available_products,
                random.randint(1, 3)
            )

            for product_id, price in selected_products:
                quantity = random.randint(1, 3)

                cursor.execute(lineitem_sql, (
                    order_id,
                    product_id,
                    quantity,
                    price
                ))

                lineitems_added += 1

    conn.commit()

except Exception:
    conn.rollback()
    raise

print("Orders added:", orders_added)
print("Line items added:", lineitems_added)

cursor.execute("SELECT COUNT(*) FROM orders")
print("Total orders:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM lineitems")
print("Total line items:", cursor.fetchone()[0])

## Employee infractions
Planted cases for Rule 8: one employee at 17 active points (termination), one at 9 (written warning), and one whose infractions have all expired. Seven others get one active infraction each.

In [ ]:
# Choose employees hired more than 500 days ago.
# This makes even the expired infractions occur after hiring.
cursor.execute("""
SELECT EmployeeID, DateHired
FROM employees
ORDER BY DateHired, EmployeeID
""")

employees = cursor.fetchall()

eligible_employees = [
    employee_id
    for employee_id, date_hired in employees
    if date_hired < TODAY - timedelta(days=500)
]

if len(eligible_employees) < 10:
    raise ValueError(
        "This plan needs 10 employees hired more than 500 days ago."
    )

termination_employee = eligible_employees[0]
warning_employee = eligible_employees[1]
expired_employee = eligible_employees[2]

# Read actual infraction IDs from the database.
cursor.execute("""
SELECT InfractionTypeID, InfractionType
FROM infractiontypes
""")

infraction_ids = {}

for infraction_id, infraction_name in cursor.fetchall():
    infraction_ids[infraction_name] = infraction_id

late_15 = infraction_ids["Late Arrival (15 min)"]
late_30 = infraction_ids["Late Arrival (30 min)"]
dress_code = infraction_ids["Dress Code Violation"]
absence = infraction_ids["Unexcused Absence"]
no_call = infraction_ids["No-Call/No-Show"]

# Each tuple contains: EmployeeID, InfractionTypeID, days ago.
infraction_plan = [
    # 17 active points: termination test.
    (termination_employee, no_call, 90),
    (termination_employee, no_call, 60),
    (termination_employee, absence, 30),

    # 9 active points: written warning test.
    (warning_employee, absence, 120),
    (warning_employee, late_30, 80),
    (warning_employee, dress_code, 40),

    # Both infractions are over a year old: 0 active points.
    (expired_employee, no_call, 450),
    (expired_employee, absence, 400),

    # Seven other employees each receive one active infraction.
    (eligible_employees[3], late_15, 20),
    (eligible_employees[4], late_30, 45),
    (eligible_employees[5], dress_code, 70),
    (eligible_employees[6], absence, 110),
    (eligible_employees[7], late_15, 160),
    (eligible_employees[8], dress_code, 220),
    (eligible_employees[9], late_30, 300)
]

rows = []

for employee_id, infraction_type_id, days_ago in infraction_plan:
    infraction_date = TODAY - timedelta(days=days_ago)

    rows.append((
        employee_id,
        infraction_type_id,
        infraction_date
    ))

sql = """
INSERT INTO employeeinfractions
(EmployeeID, InfractionTypeID, InfractionDate)
VALUES (%s, %s, %s)
"""

cursor.executemany(sql, rows)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM employeeinfractions")
print("Total infractions:", cursor.fetchone()[0])

print("Termination test — EmployeeID:", termination_employee)
print("Written warning test — EmployeeID:", warning_employee)
print("Expired infractions test — EmployeeID:", expired_employee)

## Training
Every employee takes Basic Enrollment Training within 30 days of being hired. Sales and Customer Service staff also take Customer Service Excellence, and half of all employees take Product Knowledge. The first two employees fail Basic Enrollment (Rule 5), and no trainer trains themselves. This step also moves both trainers' hire dates earlier so they were hired before anyone they train.

In [ ]:
# Read employees and identify the two trainers.
cursor.execute("""
SELECT EmployeeID, DateHired, Department
FROM employees
ORDER BY EmployeeID
""")
employees = cursor.fetchall()

trainer_ids = [
    employee_id
    for employee_id, date_hired, department in employees
    if department == "Training"
]

if len(trainer_ids) != 2:
    raise ValueError("Expected exactly two employees in Training.")

# Read course IDs by name.
cursor.execute("""
SELECT CourseID, CourseType
FROM trainingcourse
""")

course_ids = {}

for course_id, course_type in cursor.fetchall():
    course_ids[course_type] = course_id

basic_course = course_ids["Basic Enrollment Training"]
service_course = course_ids["Customer Service Excellence"]
product_course = course_ids["Product Knowledge"]

# Give Product Knowledge to half the employees.
employee_ids = [employee[0] for employee in employees]

product_trainees = random.sample(
    employee_ids,
    len(employee_ids) // 2
)

# Two intentional failures in Basic Enrollment Training.
failed_employee_ids = employee_ids[:2]

trainer_hire_date = date(TODAY.year - 22, 1, 1)

sql = """
INSERT INTO training
(CourseID, EmployeeID, TrainerID, TrainingDate, Results)
VALUES (%s, %s, %s, %s, %s)
"""

rows = []

try:
    # Adjust the sample trainers' hire dates.
    for trainer_id in trainer_ids:
        cursor.execute("""
        UPDATE employees
        SET DateHired = %s
        WHERE EmployeeID = %s
        """, (trainer_hire_date, trainer_id))

    # Read the updated dates.
    cursor.execute("""
    SELECT EmployeeID, DateHired, Department
    FROM employees
    ORDER BY EmployeeID
    """)
    employees = cursor.fetchall()

    for employee_id, date_hired, department in employees:

        # Basic training occurs within 30 days of hiring.
        basic_date = min(
            date_hired + timedelta(days=random.randint(1, 30)),
            TODAY
        )

        courses = [basic_course]

        if department in ["Sales", "Customer Service"]:
            courses.append(service_course)

        if employee_id in product_trainees:
            courses.append(product_course)

        for course_id in courses:
            if course_id == basic_course:
                training_date = basic_date
            else:
                # Other courses occur on or after Basic training.
                days_available = (TODAY - basic_date).days

                training_date = basic_date + timedelta(
                    days=random.randint(0, days_available)
                )

            trainer_id = random.choice(trainer_ids)

            # A trainer cannot train themselves.
            if trainer_id == employee_id:
                if trainer_id == trainer_ids[0]:
                    trainer_id = trainer_ids[1]
                else:
                    trainer_id = trainer_ids[0]

            result = 1

            if (
                course_id == basic_course
                and employee_id in failed_employee_ids
            ):
                result = 0

            rows.append((
                course_id,
                employee_id,
                trainer_id,
                training_date,
                result
            ))

    cursor.executemany(sql, rows)
    conn.commit()

except Exception:
    conn.rollback()
    raise

cursor.execute("SELECT COUNT(*) FROM training")
print("Total training records:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM training WHERE Results = 0")
print("Failed training records:", cursor.fetchone()[0])

cursor.execute("""
SELECT COUNT(*)
FROM training
WHERE EmployeeID = TrainerID
""")
print("Self-training records:", cursor.fetchone()[0])

print("Trainers:", trainer_ids)
print("Failed Basic Enrollment — EmployeeIDs:", failed_employee_ids)

## Realistic birth dates
Employees are created with January 1 birthdays to keep the age math simple. This step gives each one a realistic birth date while keeping their age the same and their hire date valid (at least 18 when hired, or 15 for the two minors).

In [ ]:
dob_random = random.Random(42)

def generate_dob(age):
    month = dob_random.randint(1, 12)
    day = dob_random.randint(1, 28)

    birth_year = TODAY.year - age

    # A birthday still ahead this year requires an earlier birth year.
    if (month, day) > (TODAY.month, TODAY.day):
        birth_year -= 1

    return date(birth_year, month, day)


cursor.execute("""
    SELECT
        EmployeeID,
        DateHired,
        TIMESTAMPDIFF(YEAR, DOB, CURDATE()) AS Age
    FROM employees
    ORDER BY EmployeeID
""")

employees = cursor.fetchall()
updates = []

for employee_id, date_hired, age in employees:
    minimum_hire_age = 15 if age < 18 else 18

    # Keep the generated birthday consistent with the hire date.
    for attempt in range(1000):
        new_dob = generate_dob(age)
        earliest_hire = new_dob.replace(
            year=new_dob.year + minimum_hire_age
        )

        if earliest_hire <= date_hired:
            updates.append((new_dob, employee_id))
            break
    else:
        raise ValueError(
            f"Could not generate a valid DOB for Employee {employee_id}"
        )

cursor.executemany(
    "UPDATE employees SET DOB = %s WHERE EmployeeID = %s",
    updates
)

conn.commit()
print("Employee DOBs updated:", len(updates))

## Check the row counts
Expected: 30 customers, 7 discount tiers, 25 employees, 5 infraction types, 15 infractions, 25 products, and 50 orders. Line item and training counts depend on the random draws (102 and 50 in the presentation's data).

In [ ]:
tables = [
    "customers",
    "discounttiers",
    "employeeinfractions",
    "employees",
    "infractiontypes",
    "lineitems",
    "orders",
    "products",
    "training",
    "trainingcourse"
]

for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM `{table}`")
    count = cursor.fetchone()[0]
    print(f"{table}: {count}")

## Close the connection

In [ ]:
cursor.close()
conn.close()